# k-means

`2. US_checkin_v2`의 좌표로 `k=5` 고정 `MiniBatchKMeans`를 학습하고, 클러스터 중심 좌표를 MongoDB 컬렉션에 저장한 뒤 체크인 문서와 `0. US_group_checkin_labeled`에 클러스터 id를 부여합니다.

In [14]:
from datetime import datetime, timezone

import numpy as np
from pymongo import MongoClient, UpdateOne
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm


In [15]:
HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

CHECKIN_COL = "2. US_checkin_v2"
GROUP_CHECKIN_COL = "0. US_group_checkin_labeled"
CLUSTER_INFO_COL = "3. loc_cluster_info_k5"
META_COL = "3. loc_cluster_info_k5_meta"
ASSIGN_FIELD = "loc_cluster_id_k5"

K = 5
TRAIN_BATCH = 20000
ASSIGN_BATCH = 20000
RANDOM_STATE = 42

client = MongoClient(host=HOST, port=PORT)
db = client[DB_NAME]
checkin_col = db[CHECKIN_COL]
group_checkin_col = db[GROUP_CHECKIN_COL]
cluster_info_col = db[CLUSTER_INFO_COL]
meta_col = db[META_COL]

query = {"latitude": {"$ne": None}, "longitude": {"$ne": None}}
n_points = checkin_col.count_documents(query)
n_group_points = group_checkin_col.count_documents(query)

print("db:", DB_NAME)
print("input:", CHECKIN_COL)
print("group checkin output:", GROUP_CHECKIN_COL)
print("cluster info output:", CLUSTER_INFO_COL)
print("assign field:", ASSIGN_FIELD)
print("checkin points with coordinates:", n_points)
print("group checkin points with coordinates:", n_group_points)


db: ejoow
input: 2. US_checkin_v2
group checkin output: 0. US_group_checkin_labeled
cluster info output: 3. loc_cluster_info_k5
assign field: loc_cluster_id_k5
checkin points with coordinates: 897701
group checkin points with coordinates: 0


## 1. k=5 MiniBatchKMeans 학습

In [16]:
model = MiniBatchKMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    batch_size=TRAIN_BATCH,
    n_init=10,
)

cursor = checkin_col.find(
    query,
    {"latitude": 1, "longitude": 1},
    no_cursor_timeout=True,
)

buf_xy = []
pbar = tqdm(total=n_points, desc="Training k=5 model")

for doc in cursor:
    buf_xy.append([doc["latitude"], doc["longitude"]])

    if len(buf_xy) >= TRAIN_BATCH:
        model.partial_fit(np.asarray(buf_xy, dtype=np.float32))
        pbar.update(len(buf_xy))
        buf_xy.clear()

if buf_xy:
    model.partial_fit(np.asarray(buf_xy, dtype=np.float32))
    pbar.update(len(buf_xy))

pbar.close()
cursor.close()

print("trained model with k =", model.n_clusters)
model.cluster_centers_


/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Training k=5 model:   0%|          | 0/897701 [00:00<?, ?it/s]

trained model with k = 5


array([[ 40.82885 , -73.994255],
       [ 40.825687, -73.9996  ],
       [ 40.654667, -74.00653 ],
       [ 40.772438, -73.88979 ],
       [ 40.66062 , -73.78041 ]], dtype=float32)

## 2. 클러스터 중심 좌표를 MongoDB에 저장

In [17]:
now = datetime.now(timezone.utc)
cluster_docs = []

for cluster_id, (latitude, longitude) in enumerate(model.cluster_centers_):
    cluster_docs.append({
        "_id": f"k_{K}_cluster_{cluster_id}",
        "cluster_id": int(cluster_id),
        "k": int(K),
        "latitude": float(latitude),
        "longitude": float(longitude),
        "assign_field": ASSIGN_FIELD,
        "source_collection": CHECKIN_COL,
        "model": "MiniBatchKMeans",
        "updated_at": now,
    })

for doc in cluster_docs:
    cluster_info_col.replace_one({"_id": doc["_id"]}, doc, upsert=True)

meta_doc = {
    "_id": f"k_{K}_summary",
    "k": int(K),
    "n_points": int(n_points),
    "assign_field": ASSIGN_FIELD,
    "cluster_info_collection": CLUSTER_INFO_COL,
    "centroids": model.cluster_centers_.tolist(),
    "updated_at": now,
}

meta_col.replace_one({"_id": meta_doc["_id"]}, meta_doc, upsert=True)

print("saved cluster info:", CLUSTER_INFO_COL, len(cluster_docs))
print("saved meta:", META_COL)
cluster_docs


saved cluster info: 3. loc_cluster_info_k5 5
saved meta: 3. loc_cluster_info_k5_meta


[{'_id': 'k_5_cluster_0',
  'cluster_id': 0,
  'k': 5,
  'latitude': 40.82884979248047,
  'longitude': -73.99425506591797,
  'assign_field': 'loc_cluster_id_k5',
  'source_collection': '2. US_checkin_v2',
  'model': 'MiniBatchKMeans',
  'updated_at': datetime.datetime(2026, 4, 1, 13, 24, 30, 444568, tzinfo=datetime.timezone.utc)},
 {'_id': 'k_5_cluster_1',
  'cluster_id': 1,
  'k': 5,
  'latitude': 40.825687408447266,
  'longitude': -73.99960327148438,
  'assign_field': 'loc_cluster_id_k5',
  'source_collection': '2. US_checkin_v2',
  'model': 'MiniBatchKMeans',
  'updated_at': datetime.datetime(2026, 4, 1, 13, 24, 30, 444568, tzinfo=datetime.timezone.utc)},
 {'_id': 'k_5_cluster_2',
  'cluster_id': 2,
  'k': 5,
  'latitude': 40.654666900634766,
  'longitude': -74.00653076171875,
  'assign_field': 'loc_cluster_id_k5',
  'source_collection': '2. US_checkin_v2',
  'model': 'MiniBatchKMeans',
  'updated_at': datetime.datetime(2026, 4, 1, 13, 24, 30, 444568, tzinfo=datetime.timezone.utc)},

## 3. 전체 체크인과 그룹 체크인에 클러스터 id 부여

In [18]:
def assign_cluster_ids(target_col, total_docs, desc):
    cursor = target_col.find(
        query,
        {"_id": 1, "latitude": 1, "longitude": 1},
        no_cursor_timeout=True,
    )

    ops = []
    buf_ids = []
    buf_xy = []
    pbar = tqdm(total=total_docs, desc=desc)

    try:
        for doc in cursor:
            buf_ids.append(doc["_id"])
            buf_xy.append([doc["latitude"], doc["longitude"]])

            if len(buf_ids) >= ASSIGN_BATCH:
                labels = model.predict(np.asarray(buf_xy, dtype=np.float32))
                for _id, label in zip(buf_ids, labels):
                    ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(label)}}))

                target_col.bulk_write(ops, ordered=False)
                pbar.update(len(buf_ids))
                ops.clear()
                buf_ids.clear()
                buf_xy.clear()

        if buf_ids:
            labels = model.predict(np.asarray(buf_xy, dtype=np.float32))
            for _id, label in zip(buf_ids, labels):
                ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(label)}}))
            target_col.bulk_write(ops, ordered=False)
            pbar.update(len(buf_ids))
    finally:
        pbar.close()
        cursor.close()

assign_cluster_ids(checkin_col, n_points, f"Assigning {ASSIGN_FIELD} to {CHECKIN_COL}")
assign_cluster_ids(group_checkin_col, n_group_points, f"Assigning {ASSIGN_FIELD} to {GROUP_CHECKIN_COL}")
print("done:", ASSIGN_FIELD, "updated in", CHECKIN_COL, "and", GROUP_CHECKIN_COL)


/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Assigning loc_cluster_id_k5 to 2. US_checkin_v2:   0%|          | 0/897701 [00:00<?, ?it/s]

/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Assigning loc_cluster_id_k5 to 0. US_group_checkin_labeled: 0it [00:00, ?it/s]

done: loc_cluster_id_k5 updated in 2. US_checkin_v2 and 0. US_group_checkin_labeled


## 4. 저장된 클러스터 정보 확인

In [19]:
list(cluster_info_col.find({}, {"_id": 1, "cluster_id": 1, "latitude": 1, "longitude": 1}).sort("cluster_id", 1))


[{'_id': 'k_5_cluster_0',
  'cluster_id': 0,
  'latitude': 40.82884979248047,
  'longitude': -73.99425506591797},
 {'_id': 'k_5_cluster_1',
  'cluster_id': 1,
  'latitude': 40.825687408447266,
  'longitude': -73.99960327148438},
 {'_id': 'k_5_cluster_2',
  'cluster_id': 2,
  'latitude': 40.654666900634766,
  'longitude': -74.00653076171875},
 {'_id': 'k_5_cluster_3',
  'cluster_id': 3,
  'latitude': 40.772438049316406,
  'longitude': -73.8897933959961},
 {'_id': 'k_5_cluster_4',
  'cluster_id': 4,
  'latitude': 40.660621643066406,
  'longitude': -73.78041076660156}]

## 5. `0. US_group_checkin_labeled`의 `cluster_label` distinct 값 확인

In [20]:
cluster_labels = sorted(group_checkin_col.distinct("cluster_label"))
print(f"{GROUP_CHECKIN_COL} cluster_label distinct values:", cluster_labels)
print("count:", len(cluster_labels))


0. US_group_checkin_labeled cluster_label distinct values: [0, 1, 2, 3, 4, 5]
count: 6
